# Declaring and deploying Apps

The code of the tutorial page "Declaring and deploying Apps".
Run `5_a_full_example` first, in the same folder. The app itself is in `plant_apps.py`.

In [1]:
import time
import acquirium as aq
from plant_apps import PumpPowerKW

acq = aq.init()

## 2. Check it

Runs the app on the stored data. Nothing is saved.

In [2]:
result = acq.check_app(PumpPowerKW, limit=2)

for binding in result["bindings"]:
    print(binding["row"]["pump"], binding["error"])
    print(binding["outputs"]["power"]["values"])

urn:swro/P1 None
[{'time': '2025-01-01T00:00:00+00:00', 'value': 1187.4115681745568}, {'time': '2025-01-01T01:00:00+00:00', 'value': 1181.6080107814955}]
urn:swro/P2 None
[{'time': '2025-01-01T00:00:00+00:00', 'value': 106.08985852963146}, {'time': '2025-01-01T01:00:00+00:00', 'value': 101.47394375763294}]


## 3. Deploy it

In [3]:
acq.deploy_app(PumpPowerKW)

{'ok': True, 'name': 'pump-power-kw', 'status': 'deployed'}

Wait until the app has caught up with the data.

In [4]:
while acq.list_apps()["apps"][0]["status"] != "idle":
    time.sleep(1)

acq.list_apps()["apps"][0]

{'name': 'pump-power-kw',
 'entrypoint': 'plant_apps:PumpPowerKW',
 'grouping': 'per_match',
 'status': 'idle',
 'binding_count': 2,
 'binding_statuses': {'idle': 2},
 'graph_revision': 12,
 'plan_current': True,
 'error': None}

## 4. Query the result

In [5]:
q = acq.query().measurement().where(app="pump-power-kw")

q.dataframe(limit=3)

time,data__P1 mechanical power (pump-power-kw[power]),data__P2 mechanical power (pump-power-kw[power])
"datetime[μs, UTC]",f64,f64
2025-01-01 00:00:00 UTC,1187.411568,106.089859
2025-01-01 01:00:00 UTC,1181.608011,101.473944
2025-01-01 02:00:00 UTC,1165.345244,97.750261


Insert a new reading into one of the inputs, and give the app a few seconds.

In [6]:
acq.insert_timeseries("csv-data", "P1-mechanical-power", [
    ("2026-01-01", 1_200_000.0),
])
time.sleep(8)

q.dataframe(start="2026-01-01")

time,data__P1 mechanical power (pump-power-kw[power])
"datetime[μs, UTC]",f64
2026-01-01 00:00:00 UTC,1200.0


## Changing and removing an app

Deploy again after a change of the code, and apply it to the old data.

In [7]:
acq.deploy_app(PumpPowerKW)

acq.reprocess_app(
    "pump-power-kw",
    "2025-01-01",
    "2026-01-02",
)

{'ok': True, 'name': 'pump-power-kw', 'status': 'reprocessing', 'bindings': 2}

Remove the app.

In [8]:
acq.remove_app("pump-power-kw")

{'ok': True, 'name': 'pump-power-kw', 'status': 'removed'}

Stop Acquirium.

In [9]:
aq.shutdown()